# Service Pre-mapping experiment

Build retrieval documents and query pools, fine-tune candidate encoders, compare split and no-split indexes, and inspect the runtime pre-mapper.

Default runtime configuration: `intfloat/e5-small`, dense weight 0.6, separate VALUE/FUNCTION indexes, six services per role. The dense path requires `sentence-transformers`.

**Review note:** stored outputs are retained from the supplied notebook. This review did not rerun training or generation, and execution counters were cleared.

In [ ]:
import json, math, os, random, re, sys, time, warnings

import numpy as np
import torch

warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings(
    "ignore", message="Importing from 'sentence_transformers.losses' is deprecated")

def _find_root():
    """Find the repository root from JOI_REPO_ROOT or parent directories."""
    env = os.getenv("JOI_REPO_ROOT")
    if env:
        root = os.path.abspath(env)

        if not all(os.path.isdir(os.path.join(root, d)) for d in ("gpt_mg", "premapping")):
            raise RuntimeError(
                f"JOI_REPO_ROOT={root} holds no gpt_mg/ and premapping/ - "
                "point it at the repository root, or unset it to search from here"
            )
        return root
    try:
        cur = os.getcwd()
    except OSError as e:
        cur = os.environ.get("PWD", "")
        if not os.path.isdir(cur):
            raise RuntimeError(
                "the kernel's working directory is gone - run "
                "os.chdir('<repo>/tutorials') (or set JOI_REPO_ROOT) and re-run this cell"
            ) from e
    cur = os.path.abspath(cur)
    while True:
        if all(os.path.isdir(os.path.join(cur, d)) for d in ("gpt_mg", "premapping")):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            raise RuntimeError("repository root not found - set JOI_REPO_ROOT")
        cur = parent

ROOT = _find_root()
GPT_MG = os.path.join(ROOT, "gpt_mg")
PREMAP_DIR = os.path.join(ROOT, "premapping")
os.chdir(ROOT)
for p in (GPT_MG, PREMAP_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

SEED = 2025
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FINETUNE_ALL = os.getenv("JOI_FT_ALL", "0").strip().lower() in ("1", "true", "yes")

FIX_SPLIT = os.getenv("JOI_FIX_SPLIT", "0").strip().lower() in ("1", "true", "yes")
RUN_TAG = ("all_candidates_finetuned" if FINETUNE_ALL else
           "split_fixes" if FIX_SPLIT else "default_run")
if FINETUNE_ALL and FIX_SPLIT:
    RUN_TAG += "_split_fixes"

OUT_DATA = os.path.join(PREMAP_DIR, "data", "integrated")
RESULTS_DIR = os.path.join(PREMAP_DIR, "results")
os.makedirs(OUT_DATA, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("python:", sys.version.split()[0], "| torch:", torch.__version__, "| device:", DEVICE)
print("repo root:", os.path.basename(ROOT))
print(f"run: {RUN_TAG} (fine-tune every candidate: {FINETUNE_ALL})")


python: 3.10.20 | torch: 2.6.0+cu124 | device: cuda
repo root: joilang-review-artifacts
run: default_run (fine-tune every candidate: False)


## 1. Retrieval documents

Build role-separated and merged device documents from the runtime service catalogs. Aliases are used only by the evaluation harness.

In [ ]:
CAMEL_RE = re.compile(r"(?<!^)(?=[A-Z])")

def humanize(name):
    return re.sub(r"\s+", " ", CAMEL_RE.sub(" ", str(name).replace("_", " "))).strip()

def rel(path):
    return os.path.relpath(path, ROOT)

CATALOG_VALUE_PATH = os.path.join(GPT_MG, "version0_6", "service_list_ver1.5.4_value.json")
CATALOG_FUNCTION_PATH = os.path.join(GPT_MG, "version0_6", "service_list_ver1.5.4_function.json")
with open(CATALOG_VALUE_PATH) as f:
    catalog_value = json.load(f)
with open(CATALOG_FUNCTION_PATH) as f:
    catalog_function = json.load(f)
print(f"catalog: {rel(CATALOG_VALUE_PATH)} -> {len(catalog_value)} VALUE service entries")
print(f"         {rel(CATALOG_FUNCTION_PATH)} -> {len(catalog_function)} FUNCTION service entries")

by_dev = {}
for role, catalog in (("VALUE", catalog_value), ("FUNCTION", catalog_function)):
    for e in catalog:
        by_dev.setdefault(e["device"], {"VALUE": [], "FUNCTION": []})[role].append(e)

def service_line(e):
    return f"{humanize(e['service'])}: {str(e.get('descriptor') or '').strip()}"

ALIASES = {
    "AirConditioner": ["cooler", "aircon", "AC", "air conditioning"],
    "Television": ["TV"],
    "Refrigerator": ["fridge", "freezer"],
    "Light": ["lamp", "bulb"],
    "RobotCleaner": ["vacuum", "robot vacuum"],
    "SmartPlug": ["outlet", "plug"],
    "EmailProvider": ["email", "mail"],
    "Speaker": ["music player"],
    "Curtain": ["drapes"],
    "Blind": ["blinds"],
    "DoorLock": ["door lock"],
    "WeatherProvider": ["forecast"],
}

def alias_line(dev):
    return [f"Also called: {', '.join(ALIASES[dev])}."] if dev in ALIASES else []

role_docs, merged_docs = {}, {}
for dev, roles in by_dev.items():
    d = humanize(dev)
    for role in ("VALUE", "FUNCTION"):
        if roles[role]:
            role_docs[f"{dev}::{role}"] = "\n".join(
                [f"{d} ({dev}) {role} services."] + alias_line(dev)
                + [service_line(e) for e in roles[role]])
    merged_docs[dev] = "\n".join(
        [f"{d} ({dev}) services."] + alias_line(dev)
        + [service_line(e) for e in roles["VALUE"] + roles["FUNCTION"]])
print(f"corpus: {len(role_docs)} role docs / {len(merged_docs)} merged docs "
      f"(descriptions only - no example command appears in any document)")

def name_tokens(dev):
    return set(humanize(dev).lower().split())

DEVICES = sorted(by_dev)
ALT = {}
for d in DEVICES:
    td = name_tokens(d)
    ALT[d] = frozenset({d} | {o for o in DEVICES if o != d
                             and (name_tokens(o) <= td or td <= name_tokens(o))})

for group in sorted({ALT[d] for d in DEVICES if len(ALT[d]) > 1}, key=sorted):
    print(f"  alternative group: {sorted(group)}")


catalog: gpt_mg/version0_6/service_list_ver1.5.4_value.json -> 137 VALUE service entries
         gpt_mg/version0_6/service_list_ver1.5.4_function.json -> 151 FUNCTION service entries
corpus: 84 role docs / 53 merged docs (descriptions only - no example command appears in any document)
  alternative group: ['GasValve', 'Valve']
  alternative group: ['Light', 'LightSensor']


## 2. Query pools

### 2.1 Template and paraphrase queries

Generate separate training and held-out phrasing registers from catalog metadata.

In [ ]:
def parse_enums(enum_desc_list):
    vals = []
    for item in enum_desc_list or []:
        c = str(item).replace("\u2022", "").strip()
        v = (c.split(" - ", 1)[0] if " - " in c else c.split()[0] if c.split() else "").strip()
        if v and v.lower() not in {x.lower() for x in vals}:
            vals.append(v)
    return vals

def template_value_examples(device, service, meta):
    d, s = humanize(device), humanize(service).lower()
    ret = str(meta.get("return_descriptor") or s).strip()
    ex = [f"What is the current {s} of the {d}?",
          f"Check {d} {s}",
          f"Tell me the {s} status of the {d}",
          f"Get {d} {s} info",
          f"What is the {ret.lower()} of the {d}?"]
    for v in parse_enums(meta.get("enums_descriptor"))[:4]:
        ex += [f"If the {d} {s} is {v}, send a notification",
               f"When the {d} {s} becomes {v}, alert me",
               f"Wait until the {d} {s} is {v}"]
    return ex

def template_function_examples(device, service, meta):
    d, s = humanize(device), humanize(service)
    arg_t = str(meta.get("argument_type") or "").upper()
    arg_d = str(meta.get("argument_descriptor") or s).strip().lower()
    enums = parse_enums(meta.get("enums_descriptor"))
    ex = []
    if arg_t.startswith("ENUM") and enums:
        for v in enums[:6]:
            ex += [f"Set the {d} {arg_d} to {v}", f"Change the {d} to {v}",
                   f"Switch the {d} to {v} mode"]
    elif "DOUBLE" in arg_t or "INTEGER" in arg_t:
        for v in ("20", "50", "80"):
            ex += [f"Set the {d} {arg_d} to {v}", f"Adjust the {d} {arg_d} to {v}"]
    elif "STRING" in arg_t or "BOOL" in arg_t or "BINARY" in arg_t:
        ex += [f"Set the {d} {arg_d} to hello", f"Update the {d} {arg_d}"]
    else:
        sl = s.lower()
        ex += [f"{sl} the {d.lower()}", f"Please {sl} for the {d.lower()}",
               f"Use the {d} to {sl}", f"Trigger {sl} for the {d}"]
    return ex

def paraphrase_value_examples(device, service, meta):
    d, s = humanize(device), humanize(service).lower()
    dl = d.lower()
    ex = [f"How is the {s} of the {d} right now?",
          f"Could you look up the {dl} {s} for me?",
          f"I want to know the current {s} on the {d}"]
    for v in parse_enums(meta.get("enums_descriptor"))[:2]:
        ex += [f"Whenever the {d} {s} is {v}, let me know",
               f"As soon as the {dl} goes into {v}, alert me"]
    return ex

def paraphrase_function_examples(device, service, meta):
    d, s = humanize(device), humanize(service)
    dl = d.lower()
    arg_t = str(meta.get("argument_type") or "").upper()
    enums = parse_enums(meta.get("enums_descriptor"))
    if arg_t.startswith("ENUM") and enums:
        return [f"have the {d} go to {enums[0]}", f"get the {dl} into {enums[0]}",
                *( [f"flip the {dl} over to {enums[1]}"] if len(enums) > 1 else [] )]
    if "DOUBLE" in arg_t or "INTEGER" in arg_t:
        return [f"dial the {dl} to 50", f"take the {d} down to 20"]
    sl = s.lower()
    return [f"go ahead and {sl} the {dl}", f"please {sl} the {dl} now"]

def dedupe(items):
    seen, out = set(), []
    for it in items:
        k = re.sub(r"\s+", " ", str(it)).strip().lower()
        if k and k not in seen:
            seen.add(k); out.append(re.sub(r"\s+", " ", str(it)).strip())
    return out


In [ ]:

MAX_EX_PER_SERVICE = 6
seen, tmpl_singles, para_singles = set(), [], []
for role, catalog in (("VALUE", catalog_value), ("FUNCTION", catalog_function)):
    for e in catalog:
        gen_t = template_value_examples if role == "VALUE" else template_function_examples
        gen_p = paraphrase_value_examples if role == "VALUE" else paraphrase_function_examples
        for q in dedupe(gen_t(e["device"], e["service"], e))[:MAX_EX_PER_SERVICE]:
            if q.lower() not in seen:
                seen.add(q.lower())
                tmpl_singles.append({"query": q, "device": e["device"], "role": role})
        for q in dedupe(gen_p(e["device"], e["service"], e))[:3]:
            if q.lower() not in seen:
                seen.add(q.lower())
                para_singles.append({"query": q, "device": e["device"], "role": role})

rng = np.random.default_rng(SEED)
idx = rng.permutation(len(tmpl_singles))
n_test = int(round(len(tmpl_singles) * 0.1)); n_val = n_test
tmpl_test = [tmpl_singles[int(i)] for i in idx[:n_test]]
tmpl_val = [tmpl_singles[int(i)] for i in idx[n_test:n_test + n_val]]
tmpl_train = [tmpl_singles[int(i)] for i in idx[n_test + n_val:]]

pidx = rng.permutation(len(para_singles))
para_val = [para_singles[int(i)] for i in pidx[: len(pidx) // 2]]
para_test = [para_singles[int(i)] for i in pidx[len(pidx) // 2:]]
print(f"template singles: {len(tmpl_singles)} (train {len(tmpl_train)} / val {len(tmpl_val)} / test {len(tmpl_test)})")
print(f"paraphrase singles (eval-only): {len(para_singles)} (val {len(para_val)} / test {len(para_test)})")
print("paraphrase examples:", [q["query"] for q in para_test[:3]])


template singles: 1421 (train 1137 / val 142 / test 142)
paraphrase singles (eval-only): 703 (val 351 / test 352)
paraphrase examples: ['I want to know the current color control color on the Light', 'I want to know the current tvoc measurement tvoc level on the Air Quality Detector', 'go ahead and switch on the television']


### 2.2 Natural queries

Load natural catalog examples and their condition/action/fragment audit. The answerability filter keeps queries with device-specific catalog vocabulary.

In [ ]:
NAT_CATALOG_PATH = os.path.join(PREMAP_DIR, "data", "service_list_ver1.1.7_eng.json")
ROLE_LABELS_PATH = os.path.join(PREMAP_DIR, "data", "path_controlled", "query_roles.json")
with open(NAT_CATALOG_PATH) as f:
    cat117 = json.load(f)
with open(ROLE_LABELS_PATH) as f:
    role_labels = json.load(f)["labels"]
print(f"natural commands: {rel(NAT_CATALOG_PATH)} | LLM role audit: {rel(ROLE_LABELS_PATH)}")
ROLE_OF = {"condition": "VALUE", "action": "FUNCTION"}

nat_singles, n_frag = [], 0
for dev, obj in cat117.items():
    if dev not in by_dev:
        continue
    for q in obj.get("examples", []) or []:
        q = str(q).strip()
        lab = role_labels.get(f"{dev}\t{q}", "fragment")
        if not q or lab == "fragment":
            n_frag += bool(q)
            continue
        nat_singles.append({"query": q, "device": dev, "role": ROLE_OF[lab]})

AVAILABLE_ROLES = {dev: {r for r in ("VALUE", "FUNCTION") if by_dev[dev][r]} for dev in by_dev}
n_snapped = 0
if FIX_SPLIT:
    for q in nat_singles:
        avail = AVAILABLE_ROLES[q["device"]]
        if q["role"] not in avail and len(avail) == 1:
            q["role"] = next(iter(avail))
            n_snapped += 1
print(f"role consistency: {n_snapped} natural commands relabelled to the only role "
      f"their device owns ({len(by_dev) - sum(1 for d in by_dev if len(AVAILABLE_ROLES[d]) == 2)} "
      f"of {len(by_dev)} devices are single-role)")

STOP = set("the a an is are was in on off to of for and or if when only it this that be at with as by then now me my please state".split())
REF = {dev: set(re.findall(r"[a-z]+", (humanize(dev) + " "
                + " ".join(ALIASES.get(dev, [])) + " " + merged_docs[dev]).lower()))
       for dev in DEVICES}
from collections import Counter
DF = Counter(t for dev in DEVICES for t in REF[dev])

def answerable(q, dev):

    toks = set(re.findall(r"[a-z]+", q.lower())) - STOP
    return any(t in REF[dev] and DF[t] <= 4 for t in toks)

before = len(nat_singles)
dropped = [q for q in nat_singles if not answerable(q["query"], q["device"])]
nat_singles = [q for q in nat_singles if answerable(q["query"], q["device"])]
print(f"answerability filter: {before} -> {len(nat_singles)} "
      f"({len(dropped)} context-dependent commands excluded), e.g.")
for q in dropped[:3]:
    print(f"  dropped: {q['query'][:56]:56s} (was filed under {q['device']})")

nidx = rng.permutation(len(nat_singles))
n_ntest = int(round(len(nat_singles) * 0.1)); n_nval = n_ntest
nat_test = [nat_singles[int(i)] for i in nidx[:n_ntest]]
nat_val = [nat_singles[int(i)] for i in nidx[n_ntest:n_ntest + n_nval]]
nat_train = [nat_singles[int(i)] for i in nidx[n_ntest + n_nval:]]
print(f"natural singles: {len(nat_singles)} labeled ({n_frag} fragments excluded) "
      f"-> train {len(nat_train)} / val {len(nat_val)} / test {len(nat_test)}")
for q in nat_test[:3]:
    print(f"  [{q['role']:8s}] {q['query'][:60]:60s} -> {q['device']}")


natural commands: premapping/data/service_list_ver1.1.7_eng.json | LLM role audit: premapping/data/path_controlled/query_roles.json
role consistency: 0 natural commands relabelled to the only role their device owns (22 of 53 devices are single-role)


answerability filter: 1445 -> 1057 (388 context-dependent commands excluded), e.g.
  dropped: Make it cool                                             (was filed under AirConditioner)
  dropped: Change to dehumidification mode                          (was filed under AirConditioner)
  dropped: Only let the wind come out                               (was filed under AirConditioner)
natural singles: 1057 labeled (183 fragments excluded) -> train 845 / val 106 / test 106
  [VALUE   ] Tell me if playback status changes                           -> Speaker
  [FUNCTION] Set alarm volume to high                                     -> Alarm
  [VALUE   ] What percent is the wind speed now?                          -> Fan


### 2.3 Compound queries

Combine condition and action clauses across two or three device slots. Gold labels allow capability-equivalent devices.

In [ ]:
def clause_lists(dev, para=False):
    d, dl = humanize(dev), humanize(dev).lower()
    c_list, a_list = [], []
    for e in by_dev[dev]["VALUE"]:
        s = humanize(e["service"]).lower()
        enums = parse_enums(e.get("enums_descriptor"))
        rtype = str(e.get("return_type") or "").upper()
        if para:
            if enums:
                c_list += [f"Whenever the {d} {s} is {enums[0]}",
                           f"As soon as the {dl} goes into {enums[0]}"]
            elif "DOUBLE" in rtype or "INTEGER" in rtype:
                c_list += [f"Whenever the {d} {s} passes 25"]
            else:
                c_list += [f"As soon as the {d} {s} updates"]
        else:
            if enums:
                c_list += [f"If the {d} {s} is {enums[0]}", f"If the {dl} is in {enums[0]} mode"]
                if len(enums) > 1:
                    c_list += [f"When the {d} {s} becomes {enums[1]}",
                               f"When the {dl} switches to {enums[1]}"]
            elif "DOUBLE" in rtype or "INTEGER" in rtype:
                c_list += [f"If the {d} {s} is above 25", f"When the {dl} reading goes above 25"]
            else:
                c_list += [f"When the {d} {s} changes", f"Once the {dl} state changes"]
    for e in by_dev[dev]["FUNCTION"]:
        if para:
            a_list += paraphrase_function_examples(dev, e["service"], e)[:2]
        else:
            s = humanize(e["service"])
            arg_t = str(e.get("argument_type") or "").upper()
            arg_d = str(e.get("argument_descriptor") or s).strip().lower()
            enums = parse_enums(e.get("enums_descriptor"))
            if arg_t.startswith("ENUM") and enums:
                a_list += [f"set the {d} {arg_d} to {enums[0]}", f"switch the {dl} to {enums[0]}"]
                if len(enums) > 1:
                    a_list += [f"put the {dl} in {enums[1]} mode"]
            elif "DOUBLE" in arg_t or "INTEGER" in arg_t:
                a_list += [f"set the {d} {arg_d} to 50", f"adjust the {dl} to 20"]
            else:
                a_list += [f"{s.lower()} the {dl}", f"run {s.lower()} on the {dl}"]
    return dedupe(c_list), dedupe(a_list)

crng = random.Random(SEED)
clause_train, clause_test, clause_para = {}, {}, {}
for dev in DEVICES:
    c_list, a_list = clause_lists(dev)
    pc_list, pa_list = clause_lists(dev, para=True)
    entry_tr, entry_te = {}, {}
    for role, lst in (("VALUE", c_list), ("FUNCTION", a_list)):
        lst = list(lst); crng.shuffle(lst)
        half = (len(lst) + 1) // 2
        entry_te[role], entry_tr[role] = lst[:half], lst[half:]
    clause_train[dev], clause_test[dev] = entry_tr, entry_te
    clause_para[dev] = {"VALUE": pc_list, "FUNCTION": pa_list}

def build_compound(cond_src, act_src, n_two, n_three, tag, rng_):
    conds = {d: v["VALUE"] for d, v in cond_src.items() if v["VALUE"]}
    acts = {d: v["FUNCTION"] for d, v in act_src.items() if v["FUNCTION"]}
    cond_devs, act_devs = sorted(conds), sorted(acts)
    out = []
    for i in range(n_two):
        a = rng_.choice(cond_devs); b = rng_.choice([x for x in act_devs if x != a])
        out.append({"query": f"{conds[a][i % len(conds[a])]}, {acts[b][i % len(acts[b])]}",
                    "slots": [ALT[a], ALT[b]], "kind": "2-device", "pool": tag})
    for i in range(n_three):
        a = rng_.choice(cond_devs); b, c = rng_.sample([x for x in act_devs if x != a], 2)
        out.append({"query": f"{conds[a][i % len(conds[a])]}, {acts[b][i % len(acts[b])]} "
                             f"and {acts[c][i % len(acts[c])]}",
                    "slots": [ALT[a], ALT[b], ALT[c]], "kind": "3-device", "pool": tag})
    return out

qrng = random.Random(SEED)
comp_tmpl = build_compound(clause_test, clause_test, 250, 50, "compound-template", qrng)
comp_para = build_compound(clause_para, clause_para, 250, 50, "compound-paraphrase", qrng)
print(f"compound: {len(comp_tmpl)} template-register + {len(comp_para)} paraphrase-register")
print("  e.g.", comp_para[0]["query"])
print("       slots =", [sorted(s) for s in comp_para[0]["slots"]])


compound: 300 template-register + 300 paraphrase-register
  e.g. Whenever the Contact Sensor contact sensor contact is closed, have the Dehumidifier go to cooling
       slots = [['ContactSensor'], ['Dehumidifier']]


## 3. Encoder fine-tuning

Train with `MultipleNegativesRankingLoss`. Split and no-split jobs use the same queries but different positive documents. GPU is used when available.

In [ ]:
from torch.utils.data import DataLoader
from sentence_transformers import InputExample, SentenceTransformer, losses

PARAMS = {
    "e5":           {"hf": "intfloat/e5-small",                   "epochs": 10, "batch": 32, "lr": 2e-5, "prefix": True},
    "mxbai":        {"hf": "mixedbread-ai/mxbai-embed-xsmall-v1", "epochs": 10, "batch": 32, "lr": 2e-5, "prefix": False},
    "multilingual": {"hf": "intfloat/multilingual-e5-small",      "epochs": 10, "batch": 32, "lr": 2e-5, "prefix": True},
    "bge-m3":       {"hf": "BAAI/bge-m3",                         "epochs": 10, "batch": 8,  "lr": 2e-5, "prefix": False},
}
WARMUP_RATIO, W, TOP_K = 0.1, 0.6, 10
RETRAIN = os.getenv("JOI_RETRAIN", "1").strip().lower() in ("1", "true", "yes")
CKPT_DIR = os.path.join(PREMAP_DIR, "models")

CKPT_SUFFIX = ("v7-allft" if FINETUNE_ALL else "v7") + ("-fixroles" if FIX_SPLIT else "")
FT_JOBS = ([("e5", "split"), ("e5", "nosplit"), ("mxbai", "split"),
            ("multilingual", "split"), ("bge-m3", "split")] if FINETUNE_ALL
           else [("e5", "split"), ("e5", "nosplit")])

FT_BATCH = 32
FT_LOG = []

ft_queries = ([{"query": q["query"], "device": q["device"], "role": q["role"]}
               for q in tmpl_train + nat_train]
              + [{"query": q, "device": dev, "role": role}
                 for dev, entry in clause_train.items()
                 for role in ("VALUE", "FUNCTION") for q in entry[role]])

def positive_doc(item, path):
    if path == "split":
        key = f"{item['device']}::{item['role']}"
        if key not in role_docs:
            other = "FUNCTION" if item["role"] == "VALUE" else "VALUE"
            key = f"{item['device']}::{other}"
        return role_docs.get(key)
    return merged_docs[item["device"]]

for path in ("split", "nosplit"):
    with open(os.path.join(OUT_DATA, f"finetune_pairs_{path}.jsonl"), "w", encoding="utf-8") as f:
        for it in ft_queries:
            doc = positive_doc(it, path)
            if doc:
                f.write(json.dumps({"query": it["query"], "device": it["device"],
                                    "role": it["role"], "path": path,
                                    "positive_doc": doc}, ensure_ascii=False) + "\n")

n_role_mismatch = sum(
    1 for it in ft_queries
    if (positive_doc(it, "split") or "").partition("\n")[0].endswith(f"{it['role']} services.") is False)
print(f"split path: {n_role_mismatch} / {len(ft_queries)} pairs whose positive document's "
      f"role differs from the query's role")

PAIR_FILES = [rel(os.path.join(OUT_DATA, f"finetune_pairs_{p}.jsonl"))
              for p in ("split", "nosplit")]
PROVENANCE = {
    "run": RUN_TAG,

    "written_by": "a re-execution of tutorials/premapping_finetune_tutorial.ipynb",
    "interpreter": f"python {sys.version.split()[0]}, torch device {DEVICE}",
    "regenerable_without_gpu_or_key": False,
    "needs": "Notebook re-execution and GPU training; no API key required",
    "finetune_all": FINETUNE_ALL,
    "seed": SEED,
    "corpus": {"role_documents": len(role_docs), "merged_documents": len(merged_docs),
               "built_from": [rel(CATALOG_VALUE_PATH), rel(CATALOG_FUNCTION_PATH)]},
    "training_queries": {
        "total": len(ft_queries),
        "template": len(tmpl_train),
        "natural": len(nat_train),
        "clause": len(ft_queries) - len(tmpl_train) - len(nat_train),
        "template_and_clause_from": [rel(CATALOG_VALUE_PATH), rel(CATALOG_FUNCTION_PATH)],
        "natural_from": [rel(NAT_CATALOG_PATH), rel(ROLE_LABELS_PATH)],
    },
    "recipe": {"warmup_ratio": WARMUP_RATIO, "max_seq_length": 256,
               "uniform_batch": FT_BATCH if FINETUNE_ALL else None,
               "note": ("every fine-tuned encoder sees identical queries, epochs, lr "
                        "and in-batch negatives" if FINETUNE_ALL else
                        "only e5-small, the encoder the runtime uses, is fine-tuned; "
                        "the other candidates are evaluated as released"),
               "caveat": ("hyperparameters (10 epochs, lr 2e-5, 256 tokens) were tuned "
                          "for e5-small and reused unchanged for every encoder; the "
                          "*_val splits are built but not used for per-model "
                          "selection"),
               "params": {k: {kk: vv for kk, vv in v.items() if kk != "prefix"}
                          for k, v in PARAMS.items()}},
    "split_fixes_enabled": FIX_SPLIT,
    "role_snapped_queries": n_snapped,
    "split_role_mismatched_pairs": n_role_mismatch,
    "pair_files": PAIR_FILES,
    "checkpoint_suffix": CKPT_SUFFIX,
    "finetuned_jobs": [f"{k}/{p}" for k, p in FT_JOBS],
}

print("training material provenance")
print(f"  corpus documents  : {PROVENANCE['corpus']['role_documents']} role / "
      f"{PROVENANCE['corpus']['merged_documents']} merged, built from")
for p in PROVENANCE["corpus"]["built_from"]:
    print(f"                      {p}")
print(f"  template + clause : generated from the same two catalogs")
print(f"  natural commands  : {rel(NAT_CATALOG_PATH)}")
print(f"                      roles from {rel(ROLE_LABELS_PATH)}")
print(f"  pairs written to  : {', '.join(PAIR_FILES)}")
print(f"integrated fine-tuning queries: {len(ft_queries)} "
      f"({len(tmpl_train)} template + {len(nat_train)} natural + "
      f"{len(ft_queries) - len(tmpl_train) - len(nat_train)} clause) - identical for both paths")
print(f"fine-tuning: {', '.join(f'{k}/{p}' for k, p in FT_JOBS)} "
      f"-> checkpoints premapping/models/<key>-<path>-{CKPT_SUFFIX}")
it = ft_queries[0]
print(f"same query, two paths: {it['query']!r} ({it['device']}, {it['role']})")
print("  split   ->", positive_doc(it, "split").splitlines()[0])
print("  nosplit ->", positive_doc(it, "nosplit").splitlines()[0])

def finetune(key, path):
    p = PARAMS[key]
    out = os.path.join(CKPT_DIR, f"{key}-{path}-{CKPT_SUFFIX}")
    if not RETRAIN and os.path.isdir(out):
        print(f"{key}/{path}: reusing {rel(out)}")
        return out
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    pairs = []
    for it in ft_queries:
        doc = positive_doc(it, path)
        if doc:
            qt = ("query: " + it["query"]) if p["prefix"] else it["query"]
            dt = ("passage: " + doc) if p["prefix"] else doc
            pairs.append(InputExample(texts=[qt, dt]))
    model = SentenceTransformer(p["hf"]).float()
    model.max_seq_length = min(model.max_seq_length or 256, 256)
    batch = FT_BATCH if FINETUNE_ALL else p["batch"]
    loader = DataLoader(pairs, shuffle=True, batch_size=batch)
    if FINETUNE_ALL and p["batch"] < FT_BATCH:
        loss = losses.CachedMultipleNegativesRankingLoss(model, mini_batch_size=p["batch"])
        loss_name = f"CachedMultipleNegativesRankingLoss(mini_batch_size={p['batch']})"
    else:
        loss = losses.MultipleNegativesRankingLoss(model)
        loss_name = "MultipleNegativesRankingLoss"
    steps = len(loader) * p["epochs"]
    FT_LOG.append({"job": f"{key}/{path}", "base_model": p["hf"], "pairs": len(pairs),
                   "batch": batch, "in_batch_negatives": batch - 1, "epochs": p["epochs"],
                   "lr": p["lr"], "steps": steps, "loss": loss_name,
                   "checkpoint": f"premapping/models/{key}-{path}-{CKPT_SUFFIX}"})
    t0 = time.perf_counter()
    model.fit(train_objectives=[(loader, loss)], epochs=p["epochs"],
              warmup_steps=max(10, int(WARMUP_RATIO * steps)),
              optimizer_params={"lr": p["lr"]}, output_path=out, show_progress_bar=True)
    print(f"{key}/{path}: {len(pairs)} pairs, batch {batch} ({batch-1} in-batch negatives), "
          f"{p['epochs']} epochs ({steps} steps) in {time.perf_counter()-t0:.0f}s -> {rel(out)}")
    del model; torch.cuda.empty_cache()
    return out

CKPTS = {(key, path): finetune(key, path) for key, path in FT_JOBS}
PROVENANCE["finetuning"] = FT_LOG
for job in FT_LOG:
    print(f"  {job['job']:18s} batch {job['batch']:2d} | {job['in_batch_negatives']:2d} in-batch negatives "
          f"| {job['steps']:4d} steps | {job['loss']}")


/tmp/ipykernel_1063277/3415639974.py:2: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import InputExample, SentenceTransformer, losses


split path: 140 / 2305 pairs whose positive document's role differs from the query's role
training material provenance
  corpus documents  : 84 role / 53 merged, built from
                      gpt_mg/version0_6/service_list_ver1.5.4_value.json
                      gpt_mg/version0_6/service_list_ver1.5.4_function.json
  template + clause : generated from the same two catalogs
  natural commands  : premapping/data/service_list_ver1.1.7_eng.json
                      roles from premapping/data/path_controlled/query_roles.json
  pairs written to  : premapping/data/integrated/finetune_pairs_split.jsonl, premapping/data/integrated/finetune_pairs_nosplit.jsonl
integrated fine-tuning queries: 2305 (1137 template + 845 natural + 323 clause) - identical for both paths
fine-tuning: e5/split, e5/nosplit -> checkpoints premapping/models/<key>-<path>-v7
same query, two paths: 'What is the current switch switch of the Refrigerator?' (Refrigerator, VALUE)
  split   -> Refrigerator (Refrigerator) VA

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4907.29it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

...

Step,Training Loss
500,0.538387


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.86it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.84it/s]

e5/nosplit: 2305 pairs, batch 32 (31 in-batch negatives), 10 epochs (730 steps) in 133s -> premapping/models/e5-nosplit-v7
  e5/split           batch 32 | 31 in-batch negatives |  730 steps | MultipleNegativesRankingLoss
  e5/nosplit         batch 32 | 31 in-batch negatives |  730 steps | MultipleNegativesRankingLoss


A role-mismatched natural query falls back to the device's other role document unless `JOI_FIX_SPLIT=1` is enabled.

## 4. Slot-based evaluation

Report nDCG@10 and Recall@K over device slots. The harness uses `premap.py` decomposition rules but implements its own document index and scoring path.

Harness budget: ten devices, five per role for decomposed commands. Runtime budget: twelve services, six per role. Harness metrics are not runtime coverage.

In [ ]:
from premap import SimpleBM25, _minmax, decompose_query, is_notification_clause

def slot_scores(ranked, slots, k=10):
    left = [set(s) for s in slots]
    covered_ranks = []
    for i, dev in enumerate(ranked[:k]):
        for j, s in enumerate(left):
            if s and dev in s:
                left[j] = None
                covered_ranks.append(i)
                break
    dcg = sum(1.0 / math.log2(r + 2) for r in covered_ranks)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(slots), k)))
    def recall(kk):
        return sum(1 for r in covered_ranks if r < kk) / len(slots)
    return dcg / idcg if idcg else 0.0, recall(5), recall(k)

class View:
    def __init__(self, enc, docs, hybrid, prefix):
        self.enc, self.prefix = enc, prefix
        self.keys = list(docs)
        texts = [docs[k] for k in self.keys]
        enc_texts = [("passage: " + t) if prefix else t for t in texts]
        self.embs = np.asarray(enc.encode(enc_texts, normalize_embeddings=True,
                                          show_progress_bar=False))
        self.bm25 = SimpleBM25(texts) if hybrid else None

    def encode_queries(self, queries):
        """One encoder call for a whole query list - see rank_devices()."""
        texts = [("query: " + q) if self.prefix else q for q in queries]
        return np.asarray(self.enc.encode(texts, normalize_embeddings=True,
                                          show_progress_bar=False))

    def top_devices(self, query, k, q_emb=None):
        if q_emb is None:
            q_emb = self.encode_queries([query])
        dense = (np.atleast_2d(q_emb) @ self.embs.T).ravel()
        s = _minmax(dense) if self.bm25 is None else \
            W * _minmax(dense) + (1 - W) * _minmax(self.bm25.get_scores(query))
        out, seen = [], set()
        for i in np.argsort(-s):
            dev = self.keys[int(i)].split("::")[0]
            if dev not in seen:
                seen.add(dev); out.append(dev)
            if len(out) >= k:
                break
        return out

def build_views(model_path, hybrid, prefix):
    enc = SentenceTransformer(model_path, device=DEVICE)
    v_docs = {k: v for k, v in role_docs.items() if k.endswith("::VALUE")}
    f_docs = {k: v for k, v in role_docs.items() if k.endswith("::FUNCTION")}
    return {"value": View(enc, v_docs, hybrid, prefix),
            "function": View(enc, f_docs, hybrid, prefix),
            "union": View(enc, role_docs, hybrid, prefix),
            "merged": View(enc, merged_docs, hybrid, prefix)}, enc

DISCOURSE_ONLY = {"go", "ahead", "please", "now", "then", "also", "just", "kindly",
                  "could", "would", "can", "you", "the", "a", "an", "it", "me", "my"}

def split_actions(clause):
    parts = [p.strip() for p in re.split(r"\band\b", clause)]
    parts = [p for p in parts if len(p.split()) >= 2 and not is_notification_clause(p)]
    if not FIX_SPLIT:
        return parts
    return [p for p in parts
            if not all(t in DISCOURSE_ONLY for t in re.findall(r"[a-z]+", p.lower()))]

def rank_devices(mode, views, query):
    if mode == "nosplit":
        return views["merged"].top_devices(query, TOP_K)
    q_cond, q_act = decompose_query(query)
    if (q_cond, q_act) == (query, query):
        return views["union"].top_devices(query, TOP_K)
    acts = split_actions(q_act)

    if not acts:
        return views["value"].top_devices(q_cond, TOP_K)

    per_role = TOP_K // 2

    embs = views["value"].encode_queries([q_cond] + acts)
    value_list = views["value"].top_devices(q_cond, per_role, q_emb=embs[0])
    act_lists = [views["function"].top_devices(a, per_role, q_emb=embs[i + 1])
                 for i, a in enumerate(acts)]
    func_list, seen_f = [], set()
    for r in range(per_role):
        for lst in act_lists:
            if r < len(lst) and lst[r] not in seen_f:
                seen_f.add(lst[r]); func_list.append(lst[r])

    ranked, seen = [], set()
    for r in range(TOP_K):
        for lst in (value_list, func_list):
            if r < len(lst) and lst[r] not in seen:
                seen.add(lst[r]); ranked.append(lst[r])
    return ranked[:TOP_K]

def eval_pool(mode, views, pool):
    t0 = time.perf_counter()
    nds, r5s, r10s = [], [], []
    for q in pool:
        slots = q.get("slots") or [ALT[q["device"]]]
        n, r5, r10 = slot_scores(rank_devices(mode, views, q["query"]), slots, TOP_K)
        nds.append(n); r5s.append(r5); r10s.append(r10)
    lat = (time.perf_counter() - t0) / len(pool) * 1000
    return {"nDCG@10": float(np.mean(nds)), "R@5": float(np.mean(r5s)),
            "R@10": float(np.mean(r10s)), "lat": lat, "n": len(pool)}


In [ ]:
EVAL_POOLS = {
    "template singles": tmpl_test,
    "natural singles": nat_test,
    "paraphrase singles": para_test,
    "compound-template": comp_tmpl,
    "compound-paraphrase": comp_para,
}
ALL = [q for pool in EVAL_POOLS.values() for q in pool]
print("evaluation pools:", {k: len(v) for k, v in EVAL_POOLS.items()}, "| total", len(ALL))

def _norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", str(s).lower())).strip()

TRAIN_STRINGS = {_norm(q["query"]) for q in ft_queries}

def sub_queries(query):
    subs = [query]
    q_cond, q_act = decompose_query(query)
    if (q_cond, q_act) != (query, query):
        subs += [q_cond] + split_actions(q_act)
    return [s for s in subs if s]

UNSEEN = [q for q in ALL if not any(_norm(s) in TRAIN_STRINGS for s in sub_queries(q["query"]))]
print(f"unseen-string subset: {len(UNSEEN)}/{len(ALL)} queries "
      f"({len(ALL) - len(UNSEEN)} share a retrieval sub-query with the training set)")

ARM_SPECS = [
    ("intfloat/e5-small + BM25 (split)",     "e5",           "split",   "hybrid"),
    ("intfloat/e5-small + BM25 (no split)",  "e5",           "nosplit", "hybrid"),
    ("mxbai-embed-xsmall-v1 + BM25 (split)", "mxbai",        "split",   "hybrid"),
    ("multilingual-e5-small (split)",        "multilingual", "split",   "dense-only"),
    ("BAAI/bge-m3 (split)",                  "bge-m3",       "split",   "dense-only"),
]

CROSS_SPECS = [
    ("intfloat/e5-small (split, dense-only)",     "e5",           "split",   "dense-only"),
    ("mxbai-embed-xsmall-v1 (split, dense-only)", "mxbai",        "split",   "dense-only"),
    ("multilingual-e5-small + BM25 (split)",      "multilingual", "split",   "hybrid"),
    ("BAAI/bge-m3 + BM25 (split)",                "bge-m3",       "split",   "hybrid"),
    ("intfloat/e5-small (no split, dense-only)",  "e5",           "nosplit", "dense-only"),
]
if FINETUNE_ALL:
    ARM_SPECS = ARM_SPECS + CROSS_SPECS

N_WARMUP = 20

results = []
for label, key, mode, search in ARM_SPECS:
    model_path = CKPTS.get((key, mode), PARAMS[key]["hf"])
    finetuned = (key, mode) in CKPTS
    views, enc = build_views(model_path, search == "hybrid", PARAMS[key]["prefix"])
    eval_pool(mode, views, ALL[:N_WARMUP])
    tot = eval_pool(mode, views, ALL)
    per = {name: eval_pool(mode, views, pool) for name, pool in EVAL_POOLS.items()}
    per["unseen-string subset"] = eval_pool(mode, views, UNSEEN)
    results.append({"label": label, "key": key, "mode": mode, "search": search,
                    "finetuned": finetuned,
                    "model": rel(model_path) if os.path.isdir(model_path) else model_path,
                    "total": tot, "per": per})
    print(f"{label:44s} nDCG@10={tot['nDCG@10']:.4f} R@5={tot['R@5']:.4f} "
          f"R@10={tot['R@10']:.4f} {tot['lat']:5.1f}ms "
          f"[{'fine-tuned' if finetuned else 'as released'}]")
    for name, r in per.items():
        print(f"   {name:22s} {r['nDCG@10']:.4f} / {r['R@5']:.4f}")
    del views, enc; torch.cuda.empty_cache()


evaluation pools: {'template singles': 142, 'natural singles': 106, 'paraphrase singles': 352, 'compound-template': 300, 'compound-paraphrase': 300} | total 1200


unseen-string subset: 985/1200 queries (215 share a retrieval sub-query with the training set)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4263.59it/s]

intfloat/e5-small + BM25 (split)             nDCG@10=0.9830 R@5=0.9953 R@10=1.0000   9.6ms [fine-tuned]
   template singles       1.0000 / 1.0000
   natural singles        0.9668 / 1.0000
   paraphrase singles     1.0000 / 1.0000
   compound-template      0.9932 / 1.0000
   compound-paraphrase    0.9505 / 0.9811
   unseen-string subset   0.9931 / 0.9997


...

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4312.81it/s]

multilingual-e5-small (split)                nDCG@10=0.9783 R@5=0.9919 R@10=0.9996   9.5ms [as released]
   template singles       0.9974 / 1.0000
   natural singles        0.9601 / 0.9906
   paraphrase singles     0.9979 / 1.0000
   compound-template      0.9858 / 0.9944
   compound-paraphrase    0.9453 / 0.9767
   unseen-string subset   0.9890 / 0.9975


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 19266.37it/s]

BAAI/bge-m3 (split)                          nDCG@10=0.9807 R@5=0.9925 R@10=0.9996  15.5ms [as released]
   template singles       1.0000 / 1.0000
   natural singles        0.9601 / 0.9906
   paraphrase singles     0.9990 / 1.0000
   compound-template      0.9902 / 0.9961
   compound-paraphrase    0.9479 / 0.9772
   unseen-string subset   0.9913 / 0.9981


The unseen-string subset contains queries whose evaluated sub-queries do not occur in training.

## 5. Results

Evaluate every arm on the same query pools and candidate budget. Hybrid arms use dense weight 0.6; latency excludes one warm-up pass. Full provenance is written to `premapping/results/`.

Recall@10 is near saturation because evaluation queries retain vocabulary from their gold catalog entries. nDCG@10 is the more discriminating metric here.

In [ ]:
print(f"{'Model':44s} {'nDCG@10':>8s} {'Recall@10':>10s} {'Latency':>8s}  Search      Encoder")
for r in sorted(results, key=lambda r: (r['total']['nDCG@10'], r['total']['R@10']), reverse=True):
    t = r["total"]
    print(f"{r['label']:44s} {t['nDCG@10']:8.4f} {t['R@10']:10.4f} {t['lat']:7.1f}ms  "
          f"{r['search']:10s}  {'fine-tuned' if r['finetuned'] else 'as released'}")
best = max(results, key=lambda r: (r['total']['nDCG@10'], r['total']['R@10']))
print(f"\nBEST: {best['label']}  nDCG@10={best['total']['nDCG@10']:.4f}  "
      f"R@10={best['total']['R@10']:.4f}")

summary = {
    "run": RUN_TAG,
    "device": DEVICE,
    "retrieval": {
        "w": W, "top_k": TOP_K, "warmup_queries": N_WARMUP,

        "lat_scope": ("each lat is a separate timed pass over the queries it"
                      " reports, divided by their count; total.lat times the"
                      " whole evaluation stream in one pass"),
    },
    "evaluation_pools": {k: len(v) for k, v in EVAL_POOLS.items()},
    "evaluation_total": len(ALL),
    "provenance": PROVENANCE,
    "arms": results,
}
stem = os.path.join(RESULTS_DIR, f"premap_eval_{RUN_TAG}")
with open(stem + ".json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=1)
with open(stem + ".md", "w", encoding="utf-8") as f:
    f.write(f"# Retrieval evaluation - {RUN_TAG}\n\n")
    f.write(f"- training queries: {PROVENANCE['training_queries']['total']} "
            f"({PROVENANCE['training_queries']['template']} template + "
            f"{PROVENANCE['training_queries']['natural']} natural + "
            f"{PROVENANCE['training_queries']['clause']} clause), seed {SEED}\n")
    f.write(f"- built from: {', '.join(PROVENANCE['corpus']['built_from'])}, "
            f"{PROVENANCE['training_queries']['natural_from'][0]}\n")
    f.write(f"- fine-tuned: {', '.join(PROVENANCE['finetuned_jobs'])}\n")
    f.write(f"- evaluation: {len(ALL)} held-out queries "
            f"{ {k: len(v) for k, v in EVAL_POOLS.items()} }, w = {W}, top-{TOP_K}, {DEVICE}\n\n")
    f.write("| Configuration | Encoder | nDCG@10 | Recall@5 | Recall@10 | Latency (ms) | Search |\n")
    f.write("|---|---|---|---|---|---|---|\n")
    for r in results:
        t = r["total"]
        f.write(f"| {r['label']} | {'fine-tuned' if r['finetuned'] else 'as released'} "
                f"| {t['nDCG@10']:.4f} | {t['R@5']:.4f} | {t['R@10']:.4f} "
                f"| {t['lat']:.1f} | {r['search']} |\n")
    f.write("\nPer-pool nDCG@10 / Recall@5:\n\n")
    f.write("| Configuration | " + " | ".join(EVAL_POOLS) + " |\n")
    f.write("|---" * (len(EVAL_POOLS) + 1) + "|\n")
    for r in results:
        cells = " | ".join(f"{r['per'][n]['nDCG@10']:.4f} / {r['per'][n]['R@5']:.4f}"
                           for n in EVAL_POOLS)
        f.write(f"| {r['label']} | {cells} |\n")
print("\nwrote", rel(stem + ".json"), "and", rel(stem + ".md"))


Model                                         nDCG@10  Recall@10  Latency  Search      Encoder
intfloat/e5-small + BM25 (split)               0.9830     1.0000     9.6ms  hybrid      fine-tuned
intfloat/e5-small + BM25 (no split)            0.9809     0.9981     7.7ms  hybrid      fine-tuned
BAAI/bge-m3 (split)                            0.9807     0.9996    15.5ms  dense-only  as released
mxbai-embed-xsmall-v1 + BM25 (split)           0.9798     1.0000     5.8ms  hybrid      as released
multilingual-e5-small (split)                  0.9783     0.9996     9.5ms  dense-only  as released

BEST: intfloat/e5-small + BM25 (split)  nDCG@10=0.9830  R@10=1.0000

wrote premapping/results/premap_eval_default_run.json and premapping/results/premap_eval_default_run.md


For three-device split queries, alternating role lists can place the third covered slot at rank 3, so the harness's attainable nDCG may be below 1.0 even with complete slot coverage.

`JOI_FIX_SPLIT=1` removes role-mismatched training pairs and requires content-bearing action sub-queries. Compare its output with the default run.

The default table mixes fine-tuned `e5-small` rows with released candidate encoders. `JOI_FT_ALL=1` produces the all-fine-tuned comparison.

## 6. Runtime pre-mapper

Call `premap_from_version_dir` on representative commands and inspect the selected services, decomposition, backend, fallbacks, and ranking signal.

In [ ]:
import premap

VER = os.path.join(GPT_MG, "version0_6")

QUERY = "If the light is on, turn on the air conditioner"
services, meta = premap.premap_from_version_dir(QUERY, VER)
print("backend:   ", meta["backend"], "| effective_search:", meta["effective_search"])
print("decomposed:", meta.get("decomposed_query"))
print("signal:    ", meta["retrieval_signal"], "| no_match:", meta["no_match"])
for entry in meta["selected"]:
    print("   ", entry)

_ko, ko_meta = premap.premap_from_version_dir("불이 켜지면 에어컨을 켜 줘", VER)
print("\nkorean ->  signal:", ko_meta["retrieval_signal"], "| no_match:", ko_meta["no_match"])
print("           notes: ", ko_meta["notes"])


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4846.57it/s]

backend:    e5-small+bm25(split)
decomposed: {'value': 'If the light is on', 'function': 'turn on the air conditioner'}
VALUE devices:    ['FallDetector', 'Light']
FUNCTION devices: ['AirConditioner', 'AirPurifier', 'Fan', 'Humidifier']


`no_match` identifies roles whose shortlist should not replace the full catalog. `run.py` records whether any shortlist was actually applied.

In [ ]:

from version0_6.config_loader import load_version_config

_, mi_full = load_version_config(QUERY, None, None, ".")
_, mi_pre  = load_version_config(QUERY, None, None, ".", premap_services=services)
f_sp, p_sp = mi_full["messages"][0]["content"], mi_pre["messages"][0]["content"]
print(f"system prompt: {len(f_sp):,} chars -> {len(p_sp):,} chars "
      f"({100*(1-len(p_sp)/len(f_sp)):.1f}% smaller)")

COVERAGE = os.path.join(RESULTS_DIR, "runtime_coverage.json")
if os.path.exists(COVERAGE):
    with open(COVERAGE) as f:
        cov = json.load(f)
    g, ceiling = cov["gt1"], cov["budget_ceiling"]
    print(f"services still in the prompt:    {g['pair_hits']}/{g['pair_total']}"
          f" = {g['pair_recall']}   (a ranker reading the answer, same budget:"
          f" {ceiling['pair_recall']})")
    print(f"commands keeping every service:  {g['full_coverage']}/{cov['corpus']['scored']}"
          f" = {g['full_coverage_rate']}   (same ranker: {ceiling['full_coverage_rate']})")
else:
    print("no results/runtime_coverage.json yet -"
          " run python premapping/runtime_coverage.py to measure it")


system prompt: 173,204 chars -> 60,794 chars (64.9% smaller)


The split runtime requires `2 * top_k_per_role <= max_total` because VALUE and FUNCTION scores are normalized in separate indexes.

## 7. JOILang generation

Pass the shortlist to `load_version_config` and make one chat-completions request. This cell skips when no API key is configured.

In [ ]:
API_KEY = (os.getenv("OPENAI_API_KEY_PROJ_BENCH") or os.getenv("JOI_EVAL_OPENAI_API_KEY")
           or os.getenv("JOI_V15_OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY"))
if not API_KEY:
    print("no OpenAI API key configured - skipping generation")
else:
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY)
    _, model_input = load_version_config(
        f"Generate JOI Lang code for Natural Language: {QUERY}", None, None, ".",
        premap_services=services)
    t0 = time.perf_counter()
    resp = client.chat.completions.create(**model_input)
    print(f"({time.perf_counter()-t0:.1f} s, {resp.usage.total_tokens} tokens)\n")
    print(resp.choices[0].message.content)


(5.0 s, 16301 tokens)

```json
{
  "name": "조명켜지면에어컨켜기",
  "cron": "",
  "period": -1,
  "code": "if ((#Light).switch_switch == \"on\") {\n  (#AirConditioner).switch_on()\n}"
}
```


## 8. Deployment commands

```bash
cd gpt_mg
python run.py --premap version0_6 "if the light is on, turn on the air conditioner"
python run.py --premap --premap-model ../premapping/models/e5-split-v7 version0_6 "..."
```

The released `intfloat/e5-small` encoder is the default. Use `runtime_coverage.py --model` to measure a checkpoint on the deployed retrieval path.